# ELM Neuron - inferenza Kaggle con modello pretrainato `num_memory_20`

Notebook per eseguire inferenza sui dati di test NeuronIO usando il checkpoint pretrainato con 20 variabili latenti del repository [`AaronSpieler/elmneuron`](https://github.com/AaronSpieler/elmneuron).

Su Kaggle aggiungi come input il dataset `single-neurons-as-deep-nets-nmda-test-data`. Se il notebook non ha internet, aggiungi anche una copia del repository come Kaggle Dataset; altrimenti la prima cella clona automaticamente GitHub in `/kaggle/working/elmneuron`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

import numpy as np
import torch
from tqdm.auto import tqdm

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

REPO_URL = 'https://github.com/AaronSpieler/elmneuron.git'
MODEL_NAME = 'num_memory_20'

# Percorso standard del dataset Kaggle citato nel README del repo.
# Se non esiste, una cella successiva cerca automaticamente una cartella con file '*sim__*'.
TEST_DATA_DIR = KAGGLE_INPUT / 'single-neurons-as-deep-nets-nmda-test-data' / 'Data_test'

# Lascia None se vuoi usare il clone GitHub. Se hai caricato il repo come dataset Kaggle,
# puoi impostare ad esempio: Path('/kaggle/input/elmneuron/elmneuron')
UPLOADED_REPO_DIR = None

OUTPUT_DIR = KAGGLE_WORKING / 'elmneuron_num_memory_20_inference'
SAVE_PREDICTIONS = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def looks_like_elmneuron_repo(path: Path) -> bool:
    return (path / 'src' / 'expressive_leaky_memory_neuron.py').exists() and (path / 'models' / MODEL_NAME).exists()


def resolve_repo_dir() -> Path:
    candidates = []
    if UPLOADED_REPO_DIR is not None:
        candidates.append(Path(UPLOADED_REPO_DIR))

    candidates.extend([
        KAGGLE_WORKING / 'elmneuron',
        KAGGLE_INPUT / 'elmneuron',
        KAGGLE_INPUT / 'elmneuron' / 'elmneuron',
        Path.cwd(),
        Path.cwd().parent,
    ])
    if KAGGLE_INPUT.exists():
        candidates.extend(KAGGLE_INPUT.glob('*/elmneuron'))
        candidates.extend(KAGGLE_INPUT.glob('*/*/elmneuron'))

    for candidate in candidates:
        if candidate is not None and looks_like_elmneuron_repo(candidate):
            return candidate

    clone_dir = KAGGLE_WORKING / 'elmneuron'
    if clone_dir.exists() and not looks_like_elmneuron_repo(clone_dir):
        shutil.rmtree(clone_dir)

    print(f'Clono {REPO_URL} in {clone_dir} ...')
    subprocess.run(['git', 'clone', REPO_URL, str(clone_dir)], check=True)
    return clone_dir


REPO_DIR = resolve_repo_dir()
sys.path.insert(0, str(REPO_DIR))
print('Repo:', REPO_DIR)
print('Modello:', REPO_DIR / 'models' / MODEL_NAME)

In [ ]:
from src.expressive_leaky_memory_neuron import ELM
from src.neuronio.neuronio_data_utils import get_data_files_from_folder
from src.neuronio.neuronio_eval_utils import (
    compute_test_predictions,
    filter_and_extract_core_results,
)

model_dir = REPO_DIR / 'models' / MODEL_NAME
with open(model_dir / 'model_config.json', 'r', encoding='utf-8') as f:
    model_config = json.load(f)
with open(model_dir / 'train_config.json', 'r', encoding='utf-8') as f:
    train_config = json.load(f)

print(json.dumps(model_config, indent=2))
print(json.dumps(train_config, indent=2))

In [ ]:
checkpoint_path = model_dir / 'neuronio_best_model_state.pt'

model = ELM(**model_config).to(DEVICE)
state_dict = torch.load(checkpoint_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

print('Checkpoint caricato:', checkpoint_path)
print('Parametri trainabili:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def resolve_test_data_dir(preferred_dir: Path) -> Path:
    if preferred_dir.exists():
        return preferred_dir

    candidates = []
    if KAGGLE_INPUT.exists():
        candidates.extend(p for p in KAGGLE_INPUT.rglob('Data_test') if p.is_dir())
        candidates.extend(p for p in KAGGLE_INPUT.rglob('*') if p.is_dir() and any(p.glob('*sim__*')))

    if not candidates:
        raise FileNotFoundError(
            f'Non trovo TEST_DATA_DIR={preferred_dir}. Controlla il nome del dataset Kaggle input '
            'o modifica TEST_DATA_DIR nella prima cella.'
        )

    return sorted(candidates, key=lambda p: len(str(p)))[0]


TEST_DATA_DIR = resolve_test_data_dir(TEST_DATA_DIR)
print('Test data dir:', TEST_DATA_DIR)

test_files = sorted(get_data_files_from_folder([str(TEST_DATA_DIR)]))
if not test_files:
    test_files = sorted(str(p) for p in TEST_DATA_DIR.rglob('*sim__*') if p.is_file())

print('Numero file test:', len(test_files))
for path in test_files[:5]:
    print(path)
if len(test_files) > 5:
    print('...')

In [ ]:
from src.neuronio.neuronio_data_utils import parse_sim_experiment_file

burn_in_time = int(train_config.get('burn_in_time', 150))
input_window_size = int(train_config.get('input_window_size', 500))

all_y_spikes_gt = []
all_y_spikes_hat = []
all_y_soma_gt = []
all_y_soma_hat = []
saved_prediction_files = []

for test_file in tqdm(test_files, desc='Inferenza sui file test'):
    X_test, y_spike_test, y_soma_test = parse_sim_experiment_file(test_file)

    y_spikes_gt, y_spikes_hat, y_soma_gt, y_soma_hat = compute_test_predictions(
        neuron=model,
        X_test=X_test,
        y_spike_test=y_spike_test,
        y_soma_test=y_soma_test,
        burn_in_time=burn_in_time,
        input_window_size=input_window_size,
        device=DEVICE,
    )

    all_y_spikes_gt.append(y_spikes_gt)
    all_y_spikes_hat.append(y_spikes_hat)
    all_y_soma_gt.append(y_soma_gt)
    all_y_soma_hat.append(y_soma_hat)

    if SAVE_PREDICTIONS:
        out_path = OUTPUT_DIR / (Path(test_file).name + '.predictions.npz')
        np.savez_compressed(
            out_path,
            y_spikes_gt=y_spikes_gt,
            y_spikes_hat=y_spikes_hat,
            y_soma_gt=y_soma_gt,
            y_soma_hat=y_soma_hat,
            source_file=str(test_file),
        )
        saved_prediction_files.append(str(out_path))

y_spikes_gt = np.vstack(all_y_spikes_gt)
y_spikes_hat = np.vstack(all_y_spikes_hat)
y_soma_gt = np.vstack(all_y_soma_gt)
y_soma_hat = np.vstack(all_y_soma_hat)

print('Shape spike pred:', y_spikes_hat.shape)
print('Shape soma pred:', y_soma_hat.shape)

In [ ]:
metrics = filter_and_extract_core_results(
    y_spikes_gt,
    y_spikes_hat,
    y_soma_gt,
    y_soma_hat,
    desired_FP_list=[0.0025, 0.0100],
    ignore_time_at_start_ms=500,
    verbose=True,
)

summary = {
    'repo_url': REPO_URL,
    'model_name': MODEL_NAME,
    'model_config': model_config,
    'train_config': train_config,
    'test_data_dir': str(TEST_DATA_DIR),
    'num_test_files': len(test_files),
    'test_files': test_files,
    'metrics': {k: float(v) for k, v in metrics.items()},
    'prediction_files': saved_prediction_files,
}

summary_path = OUTPUT_DIR / 'inference_summary_num_memory_20.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary['metrics'], indent=2))
print('Summary salvato in:', summary_path)

In [ ]:
import matplotlib.pyplot as plt

sim_idx = 0
t0, t1 = 500, 1500

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(y_soma_gt[sim_idx, t0:t1], label='Soma GT', linewidth=1.2)
axes[0].plot(y_soma_hat[sim_idx, t0:t1], label='Soma pred', linewidth=1.2)
axes[0].legend()
axes[0].set_ylabel('mV, centrato')

axes[1].plot(y_spikes_hat[sim_idx, t0:t1], label='Probabilita spike pred', linewidth=1.2)
spike_times = np.flatnonzero(y_spikes_gt[sim_idx, t0:t1])
if len(spike_times):
    axes[1].vlines(spike_times, 0, 1, colors='tab:red', label='Spike GT', linewidth=1.0)
axes[1].legend()
axes[1].set_xlabel('Tempo relativo (ms)')
axes[1].set_ylabel('Probabilita')

fig.tight_layout()
plot_path = OUTPUT_DIR / 'example_prediction_num_memory_20.png'
fig.savefig(plot_path, dpi=160)
print('Plot salvato in:', plot_path)